In [16]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [17]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

cache_dir = LocalFileStore("./.cache/")

splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=500,
    chunk_overlap=50,
)
loader = TextLoader("./files/01_universe_overview.txt")
docs = loader.load_and_split(text_splitter=splitter)

embeddings = OpenAIEmbeddings(
    base_url=os.getenv("OPENAI_EMBEDDING_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_EMBEDDING_MODEL"),
)

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(embeddings, cache_dir)

vectorstore = FAISS.from_documents(docs, cached_embeddings)

In [18]:
results = vectorstore.similarity_search("What is dark matter?")
results

[Document(id='4c3ca96c-26e6-46ea-be9d-8dd4afcdb64b', metadata={'source': './files/01_universe_overview.txt'}, page_content='ï»¿TITLE: Universe Overview\nTOPIC: cosmos overview\nLANG: en\nThe universe is the totality of space, time, matter, and energy. It contains everything that physically exists, from tiny subatomic particles to enormous galaxy superclusters. Modern cosmology describes the universe as dynamic rather than static: it changes over time, expands, and evolves in structure. The best-supported model for its origin and evolution is the Big Bang model.\nThe Big Bang does not describe an explosion into empty space. Instead, it describes the rapid expansion of space itself from an extremely hot, dense early state about 13.8 billion years ago. In the first fractions of a second, fundamental particles formed. Within minutes, the first light elements such as hydrogen and helium nuclei were produced in a process called Big Bang nucleosynthesis.\nAbout 380,000 years after the Big Ban

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_NAME"),
)

retriever = vectorstore.as_retriever()

def format_docs(docs):
    # Stuff method: stuffing all chunks
    return "\n\n".join(doc.page_content for doc in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question based only on the following context:\n\n{context}"),
    ("human", "{question}"),
])

# LCEL Stuff chain
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

questions = [
    "What is the Big Bang and when did it occur?",
    "What is the composition of the universe in terms of ordinary matter, dark matter, and dark energy?",
    "What are the open questions in modern cosmology?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {chain.invoke(q)}")
    print("-" * 60)

Q: What is the Big Bang and when did it occur?
A: The Big Bang is the leading scientific model for the origin and evolution of the universe. It does **not** mean an explosion into pre‑existing empty space; rather, it describes:

- An **extremely hot, dense early state** in which all space, time, matter, and energy were concentrated.
- A subsequent **rapid expansion of space itself**, during which fundamental particles formed, and within minutes the first light elements (mainly hydrogen and helium nuclei) were created (Big Bang nucleosynthesis).
- Continued expansion and cooling, eventually allowing atoms to form and structure (stars, galaxies, clusters) to develop over billions of years.

In terms of timing, the Big Bang—the start of this expansion—occurred about **13.8 billion years ago**.
------------------------------------------------------------
Q: What is the composition of the universe in terms of ordinary matter, dark matter, and dark energy?
A: The universe is commonly modeled

In [20]:
# Compare Stuff vs MapReduce

from langchain_core.runnables import RunnableLambda

# Use all docs for a fair comparison
retriever = vectorstore.as_retriever()

# --- Stuff chain ---
stuff_prompt = ChatPromptTemplate.from_messages([
    ( "system", "Answer using only the provided Universe Overview context. If not found, don't make it up:\n\n{context}",
    ),
    ("human", "{question}"),
])

stuff_chain = (
    {"context": retriever,
     "question": RunnablePassthrough()}
    | stuff_prompt
    | llm
    | StrOutputParser()
)

#MapReduce chain
map_doc_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            From this chunk of Universe Overview, extract only information relevant to the question. If none, return:''
            -------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

map_doc_chain = map_doc_prompt | llm

def map_docs(inputs):
    documents = inputs["documents"]
    question = inputs["question"]
    return "\n\n".join(
        map_doc_chain.invoke(
            {"context": doc.page_content, "question": question}
        ).content
        for doc in documents
    )

map_chain = {
    "documents": retriever,
    "question": RunnablePassthrough(),
} | RunnableLambda(map_docs)

# Reduce step: take the relevant text from all chunks and answer the question
final_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Combine extracted evidence from all chunks to answer the question using only that evidence. If not enough evidence, say: I don't know based on the provided context.
            ------
            {context}
            """,
        ),
        ("human", "{question}"),
    ]
)

mapreduce_chain = {"context": map_chain, "question": RunnablePassthrough()} | final_prompt | llm

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

test_question = "What is the composition of the universe in terms of ordinary matter, dark matter, and dark energy?"

# Stuff
with get_usage_metadata_callback() as stuff_usage:
    stuff_answer = stuff_chain.invoke(test_question)

# MapReduce
with get_usage_metadata_callback() as mapreduce_usage:
    mapreduce_answer = mapreduce_chain.invoke(test_question)

STUFF Score:
SCORE: 10/10  
Reason: The answer accurately states the approximate percentages of ordinary matter, dark matter, and dark energy, notes that they are approximate, and mentions dependence on observations and models. It is complete and clearly phrased, fully matching the provided Universe Overview content.


[STUFF]
Answer: The universe’s composition is approximately:

- **About 5%** ordinary (baryonic) matter  
- **About 27%** dark matter  
- **About 68%** dark energy  

These percentages can vary slightly depending on the observational data and cosmological model used.
STUFF Token info: {'gpt-5.1-2025-11-13': {'input_tokens': 1029, 'output_tokens': 68, 'total_tokens': 1097, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}
MAPREDUCE Score:
SCORE: 10/10  
Reason: Accurately reproduces the composition percentages (5% ordinary matter, 27% dark matter, 68% dark energy), matches the context, and is clear and complete fo

In [25]:
comparison_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert evaluator. Compare two answers to the same question about Universe Overview.

QUESTION: {question}

CONTEXT (source document):
{context}

---

ANSWER A (Stuff method):
{answer_a}

---

ANSWER B (MapReduce method):
{answer_b}

---

EVALUATION CRITERIA:
1. Accuracy: Are the facts correct according to the document?
2. Completeness: Does it cover all aspects of the question?
3. Clarity: Is it well-organized and easy to understand?
4. Groundedness: Does it stick to the source or add external information?

---

RESPOND IN THIS FORMAT:

BETTER ANSWER: [A or B]

CRITERION BREAKDOWN:
- Accuracy: [A/B is better because...]
- Completeness: [A/B is better because...]
- Clarity: [A/B is better because...]
- Groundedness: [A/B is better because...]

CONCLUSION: [Why you chose A or B]

QUALITY SCORES:
- Answer A: [X]/10
- Answer B: [X]/10
"""),
    ("human", "Compare these two answers"),
])

comp_chain = comparison_prompt | llm | StrOutputParser()

result = comp_chain.invoke({
    "question": test_question,
    "context": "\n\n".join([d.page_content for d in docs]),
    "answer_a": stuff_answer,
    "answer_b": mapreduce_answer,
})

print("=" * 70)
print("PAIRWISE COMPARISON - STUFF vs MAPREDUCE")
print("=" * 70)
print(result)

PAIRWISE COMPARISON - STUFF vs MAPREDUCE
BETTER ANSWER: A

CRITERION BREAKDOWN:
- Accuracy: Both A and B give exactly the same percentages (≈5% ordinary matter, 27% dark matter, 68% dark energy), which match the source. They are equally accurate.
- Completeness: A is slightly more complete because it adds that “These percentages can vary slightly depending on the observational data and cosmological model used,” which is explicitly in the source text. B omits this nuance.
- Clarity: Both are very clear and easy to read, but A uses a brief introductory phrase (“The universe’s composition is approximately:”) and bullet list, making it marginally more polished as a standalone answer. B is also clear, just slightly more cluttered by the metadata-style wrapper.
- Groundedness: A is fully grounded in the source and includes the variability note from the document. B is also grounded but leaves out that explicit qualification.

CONCLUSION: Answer A is better because it is equally accurate but s

In [32]:
print("\n[STUFF]")
print(f"Answer: {stuff_answer}")
print(f"STUFF Token info: {stuff_usage.usage_metadata}")


print("\n[MAPREDUCE]")
print(f"Answer: {mapreduce_answer}")
print(f"MAPREDUCE Token info: {mapreduce_usage.usage_metadata}")
print("Use MAPREDUCE when: document is large, need full coverage across all sections.")
print()
print("Use STUFF when: document is small, fast + cheap response needed.")
print("Use MAPREDUCE when: document is large, need full coverage across all sections.")
print()



[STUFF]
Answer: The universe’s composition is approximately:

- **About 5%** ordinary (baryonic) matter  
- **About 27%** dark matter  
- **About 68%** dark energy  

These percentages can vary slightly depending on the observational data and cosmological model used.
STUFF Token info: {'gpt-5.1-2025-11-13': {'input_tokens': 1029, 'output_tokens': 68, 'total_tokens': 1097, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}

[MAPREDUCE]
Answer: content='Based on the provided context, the universe is composed of approximately:\n\n- **5%** ordinary (baryonic) matter  \n- **27%** dark matter  \n- **68%** dark energy' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 115, 'total_tokens': 166, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0